## 0) Variable meanings / architecture description

This notebook predicts **full spatial fire-probability maps** one hour ahead using a ConvGRU + U-Net architecture that processes padded 2D grids.

- **Prediction target**: per-pixel GOES fire confidence at `t+1`, binarized at `POSITIVE_THRESHOLD`.
- **Input**: 6 consecutive hours of 8-channel grids (GOES confidence, 5 weather variables, wind direction sin/cos, validity mask), zero-padded to 112×192.
- **Architecture**: U-Net encoder (shared across timesteps) → ConvGRU at bottleneck (14×24) → U-Net decoder with skip connections from last timestep → 1×1 output.
- **Loss**: 0.7 × FocalLoss(γ=2.0, α=0.75) + 0.3 × DiceLoss, computed only on valid (non-padding) pixels.
- **Training**: AdamW with warmup + cosine decay, gradient clipping, fire-order shuffling between epochs.
- **Evaluation**: fire-level holdout, per-fire metric breakdown, spatial prediction visualization, PR curve analysis.

## 1) Config

In [1]:
from pathlib import Path

FIRE_SELECTION = "all"
TRAIN_FIRES = "auto"
TEST_FIRES = "auto"
FIRE_TRAIN_FRACTION = 0.70
FIRE_SPLIT_SEED = 42

POSITIVE_THRESHOLD = 0.10
CLASSIFICATION_PROB_THRESHOLD = 0.50

# Grid padding target (divisible by 8, fits max 105x187)
PAD_H = 112
PAD_W = 192

# ConvGRU U-Net architecture
SEQ_LEN = 6
ENCODER_CHANNELS = [32, 64, 128]
BOTTLENECK_CHANNELS = 256
IN_CHANNELS = 8  # 7 physical + 1 validity mask

# Training
SEED = 1337
EPOCHS = 15
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
LR_WARMUP_FRACTION = 0.1

# Loss
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0
FOCAL_WEIGHT = 0.7
DICE_WEIGHT = 0.3

# RTMA config (no discounted rain for grid model)
INCLUDE_DISCOUNTED_RAIN_FEATURE = False
DISCOUNTED_RAIN_LOOKBACK_HOURS = 0
DISCOUNTED_RAIN_HALF_LIFE_DAYS = 0.0

# Section toggles
RUN_FIRE_DISCOVERY_SECTION = True
RUN_NORMALIZATION_SECTION = True
RUN_TRAINING_SECTION = True
RUN_EVALUATION_SECTION = True
RUN_VISUALIZATION_SECTION = True
RUN_PR_SECTION = True
RUN_REPORT_SECTION = True

In [2]:
print("fire selection:", FIRE_SELECTION)
print("positive threshold:", POSITIVE_THRESHOLD)
print("classification prob threshold:", CLASSIFICATION_PROB_THRESHOLD)
print(f"grid padding: {PAD_H}x{PAD_W}")
print()
print("ConvGRU U-Net config:")
print("  seq_len:", SEQ_LEN)
print("  in_channels:", IN_CHANNELS)
print("  encoder_channels:", ENCODER_CHANNELS)
print("  bottleneck_channels:", BOTTLENECK_CHANNELS)
print("  epochs:", EPOCHS)
print("  batch_size:", BATCH_SIZE)
print("  grad_accum_steps:", GRAD_ACCUM_STEPS)
print("  learning_rate:", LEARNING_RATE)
print("  grad_clip_norm:", GRAD_CLIP_NORM)
print()
print("Loss config:")
print(f"  {FOCAL_WEIGHT} * Focal(alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}) + {DICE_WEIGHT} * Dice")

fire selection: all
positive threshold: 0.1
classification prob threshold: 0.5
grid padding: 112x192

ConvGRU U-Net config:
  seq_len: 6
  in_channels: 8
  encoder_channels: [32, 64, 128]
  bottleneck_channels: 256
  epochs: 15
  batch_size: 4
  grad_accum_steps: 1
  learning_rate: 0.0003
  grad_clip_norm: 1.0

Loss config:
  0.7 * Focal(alpha=0.75, gamma=2.0) + 0.3 * Dice


## 2) Imports + setup

In [3]:
import json
import random
import sys
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

cwd = Path.cwd().resolve()
for candidate in [cwd] + list(cwd.parents):
    if (candidate / "scripts").exists() and (candidate / "docs").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not find repo root.")

from scripts.neighbor_cell_logreg import (
    discover_fire_entries,
    find_repo_root,
    iter_aligned_hours_for_fire,
    load_fire_entry_context,
    safe_divide,
    select_fire_entries,
    split_fire_entries,
    split_validation_fire_entries,
)
from scripts.neighbor_cell_nn import pick_device

In [4]:
REPO_ROOT = find_repo_root(Path.cwd())
DEFAULT_DEVICE = pick_device()
print("repo root:", REPO_ROOT)
print("device:", DEFAULT_DEVICE)
print("torch:", torch.__version__)

repo root: /Users/seanmay/Desktop/Current Projects/wildfire-prediction
device: mps
torch: 2.5.1


## 3) Fire discovery + train/test split

In [5]:
all_fire_entries = []
fire_entries = []
train_fire_entries = []
test_fire_entries = []

if RUN_FIRE_DISCOVERY_SECTION:
    all_fire_entries = discover_fire_entries(REPO_ROOT)
    fire_entries = select_fire_entries(all_fire_entries, FIRE_SELECTION)
    train_fire_entries, test_fire_entries = split_fire_entries(
        fire_entries, TRAIN_FIRES, TEST_FIRES, FIRE_TRAIN_FRACTION, FIRE_SPLIT_SEED,
    )
    print("Total fires:", len(fire_entries))
    print("Train fires:", [e["fire_name"] for e in train_fire_entries])
    print("Test fires:", [e["fire_name"] for e in test_fire_entries])

Total fires: 28
Train fires: ['Antelope', 'Bobcat', 'Caldor', 'Creek', 'Dixie', 'Glass', 'July_Complex', 'KNP_Complex', 'Kincade', 'McFarland', 'Monument', 'North_Complex', 'Red_Salmon_Complex', 'River_Complex', 'SCU_Lightning_Complex', 'SQF_Complex', 'Slater_and_Devil', 'Tamarack', 'W-5_Cold_Springs', 'Zogg']
Test fires: ['August_Complex', 'Beckwourth_Complex', 'CZU_Lightning_Complex', 'Dolan', 'LNU_Lightning_Complex', 'McCash', 'Walker', 'Windy']


## 4) Dataset statistics

Scan training fires to count total grid hours, pixels, and positive rate for the spatial grid framing.

In [6]:
ITERATION_KWARGS = {
    "discounted_rain_lookback_hours": DISCOUNTED_RAIN_LOOKBACK_HOURS,
    "discounted_rain_half_life_days": DISCOUNTED_RAIN_HALF_LIFE_DAYS,
}


def count_grid_stats(entries):
    total_hours = 0
    total_pixels = 0
    total_pos = 0
    for entry in entries:
        ctx = load_fire_entry_context(entry, REPO_ROOT)
        goes_conf = ctx["goes_conf"]
        H, W = ctx["goes_shape"]
        for t, _ in iter_aligned_hours_for_fire(
            REPO_ROOT, goes_conf, ctx["goes_time_index"],
            ctx["rtma_manifest"], ctx["rtma_manifest_path"],
            ctx["goes_shape"], ctx["goes_transform"], ctx["goes_crs"],
            include_discounted_rain=False,
            **ITERATION_KWARGS,
        ):
            total_hours += 1
            target = goes_conf[t + 1]
            valid = np.isfinite(target)
            total_pixels += int(valid.sum())
            total_pos += int((target[valid] >= POSITIVE_THRESHOLD).sum())
    return {"hours": total_hours, "pixels": total_pixels, "positives": total_pos}


train_grid_stats = count_grid_stats(train_fire_entries)
test_grid_stats = count_grid_stats(test_fire_entries)

print(f"Train: {train_grid_stats['hours']:,} hours, {train_grid_stats['pixels']:,} pixels, "
      f"{train_grid_stats['positives']:,} pos ({train_grid_stats['positives']/max(train_grid_stats['pixels'],1):.4%})")
print(f"Test:  {test_grid_stats['hours']:,} hours, {test_grid_stats['pixels']:,} pixels, "
      f"{test_grid_stats['positives']:,} pos ({test_grid_stats['positives']/max(test_grid_stats['pixels'],1):.4%})")

Train: 16,277 hours, 105,275,111 pixels, 307,417 pos (0.2920%)
Test:  5,973 hours, 45,859,549 pixels, 125,969 pos (0.2747%)


## 5) Channel normalization

Per-channel z-score computed from training fire grids (7 physical channels). The validity mask channel is not normalized.

In [7]:
CHANNEL_NAMES = [
    "goes_conf", "temperature", "wind_speed",
    "wind_dir_sin", "wind_dir_cos", "specific_humidity",
    "precipitation_1h",
]
N_PHYSICAL_CHANNELS = len(CHANNEL_NAMES)


def compute_channel_stats(entries):
    chan_sum = np.zeros(N_PHYSICAL_CHANNELS, dtype=np.float64)
    chan_sq_sum = np.zeros(N_PHYSICAL_CHANNELS, dtype=np.float64)
    chan_count = np.zeros(N_PHYSICAL_CHANNELS, dtype=np.int64)

    for entry in entries:
        ctx = load_fire_entry_context(entry, REPO_ROOT)
        goes_conf = ctx["goes_conf"]

        for t, rtma_hour in iter_aligned_hours_for_fire(
            REPO_ROOT, goes_conf, ctx["goes_time_index"],
            ctx["rtma_manifest"], ctx["rtma_manifest_path"],
            ctx["goes_shape"], ctx["goes_transform"], ctx["goes_crs"],
            include_discounted_rain=False,
            **ITERATION_KWARGS,
        ):
            conf = goes_conf[t].astype(np.float64)
            tmp = rtma_hour["TMP"].astype(np.float64)
            wind = rtma_hour["WIND"].astype(np.float64)
            wdir_rad = np.deg2rad(rtma_hour["WDIR"].astype(np.float64))
            wdir_sin = np.sin(wdir_rad)
            wdir_cos = np.cos(wdir_rad)
            spfh = rtma_hour["SPFH"].astype(np.float64)
            precip = rtma_hour["ACPC01"].astype(np.float64)

            channels = [conf, tmp, wind, wdir_sin, wdir_cos, spfh, precip]
            for i, ch in enumerate(channels):
                valid = np.isfinite(ch)
                vals = ch[valid]
                chan_sum[i] += vals.sum()
                chan_sq_sum[i] += (vals ** 2).sum()
                chan_count[i] += vals.size

    means = chan_sum / np.maximum(chan_count, 1)
    variances = np.maximum(chan_sq_sum / np.maximum(chan_count, 1) - means ** 2, 0.0)
    stds = np.sqrt(variances)
    stds_safe = np.where(stds > 0, stds, 1.0)
    return means, stds_safe


if RUN_NORMALIZATION_SECTION:
    channel_means, channel_stds = compute_channel_stats(train_fire_entries)
    for i, name in enumerate(CHANNEL_NAMES):
        print(f"  {name:>20s}: mean={channel_means[i]:.4f}, std={channel_stds[i]:.4f}")
else:
    channel_means = np.zeros(N_PHYSICAL_CHANNELS, dtype=np.float64)
    channel_stds = np.ones(N_PHYSICAL_CHANNELS, dtype=np.float64)

             goes_conf: mean=0.0024, std=0.0468
           temperature: mean=20.5692, std=7.6271
            wind_speed: mean=2.3878, std=1.7622
          wind_dir_sin: mean=-0.1080, std=0.7161
          wind_dir_cos: mean=-0.0792, std=0.6850
     specific_humidity: mean=0.0058, std=0.0024
      precipitation_1h: mean=2.6816, std=160.3612


## 6) Architecture definition

- **EncoderBlock**: two Conv3x3+BN+ReLU layers, then MaxPool2x2
- **ConvGRUCell**: convolutional GRU operating at bottleneck resolution (14×24)
- **DecoderBlock**: Upsample + concat skip + two Conv3x3+BN+ReLU layers
- **ConvGRUUNet**: full model wiring encoder, ConvGRU, and decoder
- **FocalLoss + DiceLoss**: combined loss for extreme class imbalance

In [8]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class EncoderLevel(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        features = self.conv(x)
        down = self.pool(features)
        return features, down


class DecoderLevel(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = ConvBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ConvGRUCell(nn.Module):
    def __init__(self, input_ch, hidden_ch, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.hidden_ch = hidden_ch
        self.gates = nn.Conv2d(input_ch + hidden_ch, 2 * hidden_ch, kernel_size, padding=pad, bias=True)
        self.candidate = nn.Conv2d(input_ch + hidden_ch, hidden_ch, kernel_size, padding=pad, bias=True)

    def forward(self, x, h):
        combined = torch.cat([x, h], dim=1)
        gates = torch.sigmoid(self.gates(combined))
        r, z = gates.chunk(2, dim=1)
        candidate = torch.tanh(self.candidate(torch.cat([x, r * h], dim=1)))
        h_new = (1 - z) * h + z * candidate
        return h_new


class ConvGRUUNet(nn.Module):
    def __init__(self, in_channels=8, encoder_channels=(32, 64, 128), bottleneck_ch=256):
        super().__init__()
        enc_chs = list(encoder_channels)

        # Encoder levels
        self.enc0 = EncoderLevel(in_channels, enc_chs[0])
        self.enc1 = EncoderLevel(enc_chs[0], enc_chs[1])
        self.enc2 = EncoderLevel(enc_chs[1], enc_chs[2])

        # Bottleneck
        self.bottleneck = ConvBlock(enc_chs[2], bottleneck_ch)

        # Temporal core: ConvGRU at bottleneck resolution
        self.conv_gru = ConvGRUCell(bottleneck_ch, bottleneck_ch, kernel_size=3)

        # Decoder levels
        self.dec2 = DecoderLevel(bottleneck_ch, enc_chs[2], enc_chs[2])
        self.dec1 = DecoderLevel(enc_chs[2], enc_chs[1], enc_chs[1])
        self.dec0 = DecoderLevel(enc_chs[1], enc_chs[0], enc_chs[0])

        # Output
        self.out_conv = nn.Conv2d(enc_chs[0], 1, 1)

    def forward(self, x_seq):
        """x_seq: (B, T, C, H, W) -> logits (B, 1, H, W)"""
        B, T, C, H, W = x_seq.shape

        # Initialize GRU hidden state
        h_height = H // 8
        h_width = W // 8
        h = torch.zeros(B, self.conv_gru.hidden_ch, h_height, h_width,
                        device=x_seq.device, dtype=x_seq.dtype)

        last_skip0 = None
        last_skip1 = None
        last_skip2 = None

        for t_idx in range(T):
            frame = x_seq[:, t_idx]  # (B, C, H, W)

            skip0, down0 = self.enc0(frame)    # skip0: (B, 32, H, W)
            skip1, down1 = self.enc1(down0)    # skip1: (B, 64, H/2, W/2)
            skip2, down2 = self.enc2(down1)    # skip2: (B, 128, H/4, W/4)

            bottleneck = self.bottleneck(down2)  # (B, 256, H/8, W/8)
            h = self.conv_gru(bottleneck, h)

            if t_idx == T - 1:
                last_skip0 = skip0
                last_skip1 = skip1
                last_skip2 = skip2

        # Decoder with skip connections from last timestep
        up2 = self.dec2(h, last_skip2)
        up1 = self.dec1(up2, last_skip1)
        up0 = self.dec0(up1, last_skip0)

        logits = self.out_conv(up0)  # (B, 1, H, W)
        return logits


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets, mask):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal = alpha_t * (1 - p_t) ** self.gamma * bce
        return (focal * mask).sum() / mask.sum().clamp(min=1)


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets, mask):
        probs = torch.sigmoid(logits) * mask
        targets_m = targets * mask
        intersection = (probs * targets_m).sum()
        union = probs.sum() + targets_m.sum()
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice


class CombinedLoss(nn.Module):
    def __init__(self, focal_weight=0.7, dice_weight=0.3, focal_alpha=0.75, focal_gamma=2.0):
        super().__init__()
        self.focal = FocalLoss(focal_alpha, focal_gamma)
        self.dice = DiceLoss()
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets, mask):
        return self.focal_weight * self.focal(logits, targets, mask) + \
               self.dice_weight * self.dice(logits, targets, mask)


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps):
        self.optimizer = optimizer
        self.warmup_steps = max(warmup_steps, 1)
        self.total_steps = max(total_steps, warmup_steps + 1)
        self.base_lr = optimizer.param_groups[0]["lr"]
        self.step_count = 0

    def step(self):
        self.step_count += 1
        if self.step_count <= self.warmup_steps:
            lr = self.base_lr * self.step_count / self.warmup_steps
        else:
            progress = (self.step_count - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            lr = self.base_lr * 0.5 * (1 + np.cos(np.pi * min(progress, 1.0)))
        for pg in self.optimizer.param_groups:
            pg["lr"] = lr

    def get_lr(self):
        return self.optimizer.param_groups[0]["lr"]


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("Architecture classes defined.")

Architecture classes defined.


## 7) Data pipeline

`iter_fire_grid_sequences()` streams through each fire's aligned hours with a rolling buffer of `SEQ_LEN` frames. Each frame is an 8-channel grid (7 physical channels + validity mask), padded to `PAD_H × PAD_W`. Yields `(X_seq, y_target, valid_mask)` for contiguous windows.

In [9]:
def build_frame(goes_conf_t, rtma_hour, goes_shape, channel_means, channel_stds):
    """Build an 8-channel padded frame for one timestep.
    Returns: (C, PAD_H, PAD_W) float32 array.
    """
    H, W = goes_shape
    conf = goes_conf_t.astype(np.float32)
    tmp = rtma_hour["TMP"].astype(np.float32)
    wind = rtma_hour["WIND"].astype(np.float32)
    wdir_rad = np.deg2rad(rtma_hour["WDIR"].astype(np.float32))
    wdir_sin = np.sin(wdir_rad)
    wdir_cos = np.cos(wdir_rad)
    spfh = rtma_hour["SPFH"].astype(np.float32)
    precip = rtma_hour["ACPC01"].astype(np.float32)

    raw_channels = [conf, tmp, wind, wdir_sin, wdir_cos, spfh, precip]

    # NaN -> 0 and normalize
    norm_channels = []
    for i, ch in enumerate(raw_channels):
        ch = np.nan_to_num(ch, nan=0.0)
        ch = (ch - channel_means[i]) / channel_stds[i]
        norm_channels.append(ch.astype(np.float32))

    # Validity mask: 1.0 inside grid, 0.0 in padding
    validity = np.ones((H, W), dtype=np.float32)

    # Stack and pad
    frame = np.zeros((IN_CHANNELS, PAD_H, PAD_W), dtype=np.float32)
    for i, ch in enumerate(norm_channels):
        frame[i, :H, :W] = ch
    frame[N_PHYSICAL_CHANNELS, :H, :W] = validity  # channel 7 = validity mask

    return frame


def build_target(goes_conf_t1, goes_shape):
    """Build binary target and valid mask, padded.
    Returns: target (PAD_H, PAD_W), valid_mask (PAD_H, PAD_W)
    """
    H, W = goes_shape
    raw = goes_conf_t1.astype(np.float32)
    valid_raw = np.isfinite(raw)
    raw = np.nan_to_num(raw, nan=0.0)
    binary = (raw >= POSITIVE_THRESHOLD).astype(np.float32)

    target = np.zeros((PAD_H, PAD_W), dtype=np.float32)
    target[:H, :W] = binary

    mask = np.zeros((PAD_H, PAD_W), dtype=np.float32)
    mask[:H, :W] = valid_raw.astype(np.float32)

    return target, mask


def iter_fire_grid_sequences(entry, repo_root, channel_means, channel_stds, seq_len):
    """Yield (X_seq, y_target, valid_mask) for contiguous seq_len-hour windows.
    X_seq: (seq_len, C, PAD_H, PAD_W)
    y_target: (PAD_H, PAD_W)
    valid_mask: (PAD_H, PAD_W)
    """
    ctx = load_fire_entry_context(entry, repo_root)
    goes_conf = ctx["goes_conf"]
    goes_shape = ctx["goes_shape"]

    buffer = []  # list of (t, frame)

    for t, rtma_hour in iter_aligned_hours_for_fire(
        repo_root, goes_conf, ctx["goes_time_index"],
        ctx["rtma_manifest"], ctx["rtma_manifest_path"],
        goes_shape, ctx["goes_transform"], ctx["goes_crs"],
        include_discounted_rain=False,
        **ITERATION_KWARGS,
    ):
        frame = build_frame(goes_conf[t], rtma_hour, goes_shape, channel_means, channel_stds)
        buffer.append((t, frame))

        if len(buffer) > seq_len:
            buffer.pop(0)

        if len(buffer) == seq_len:
            t_vals = [b[0] for b in buffer]
            if t_vals[-1] - t_vals[0] != seq_len - 1:
                continue

            X_seq = np.stack([b[1] for b in buffer], axis=0)  # (T, C, H, W)
            last_t = t_vals[-1]
            y_target, valid_mask = build_target(goes_conf[last_t + 1], goes_shape)
            yield X_seq, y_target, valid_mask


# Quick sanity check
if RUN_FIRE_DISCOVERY_SECTION:
    _test_entry = train_fire_entries[0]
    _count = 0
    for X_seq, y_target, valid_mask in iter_fire_grid_sequences(
        _test_entry, REPO_ROOT, channel_means, channel_stds, SEQ_LEN,
    ):
        _count += 1
        if _count == 1:
            print(f"First sequence shape: X={X_seq.shape}, y={y_target.shape}, mask={valid_mask.shape}")
            print(f"  y positives: {(y_target * valid_mask).sum():.0f}, valid pixels: {valid_mask.sum():.0f}")
        if _count >= 3:
            break
    print(f"Sampled {_count} sequences from {_test_entry['fire_name']}")

First sequence shape: X=(6, 8, 112, 192), y=(112, 192), mask=(112, 192)
  y positives: 8, valid pixels: 6206
Sampled 3 sequences from Antelope


## 8) Training loop

- Fire-order shuffled each epoch
- Streaming sequences from each fire (rolling buffer, no full-fire caching)
- Mini-batches of `BATCH_SIZE` grids with optional gradient accumulation
- Warmup + cosine decay LR schedule
- Gradient clipping for ConvGRU stability

In [10]:
set_seed(SEED)

model = ConvGRUUNet(
    in_channels=IN_CHANNELS,
    encoder_channels=ENCODER_CHANNELS,
    bottleneck_ch=BOTTLENECK_CHANNELS,
).to(DEFAULT_DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = CombinedLoss(
    focal_weight=FOCAL_WEIGHT, dice_weight=DICE_WEIGHT,
    focal_alpha=FOCAL_ALPHA, focal_gamma=FOCAL_GAMMA,
)

# Estimate total optimizer steps for LR schedule
est_seqs_per_epoch = train_grid_stats["hours"] - SEQ_LEN * len(train_fire_entries)
est_batches_per_epoch = max(est_seqs_per_epoch // BATCH_SIZE, 1)
est_steps_per_epoch = max(est_batches_per_epoch // GRAD_ACCUM_STEPS, 1)
estimated_total_steps = EPOCHS * est_steps_per_epoch
warmup_steps = int(LR_WARMUP_FRACTION * estimated_total_steps)
scheduler = WarmupCosineScheduler(optimizer, warmup_steps, estimated_total_steps)

n_params = count_parameters(model)
print(f"Parameters: {n_params:,}")
print(f"Estimated steps/epoch: {est_steps_per_epoch:,}, total: {estimated_total_steps:,}, warmup: {warmup_steps:,}")

Parameters: 5,489,441
Estimated steps/epoch: 4,039, total: 60,585, warmup: 6,058


In [ ]:
training_history = []

if RUN_TRAINING_SECTION:
    ckpt_dir = REPO_ROOT / "data" / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        epoch_steps = 0
        tp = fp = fn = tn = 0
        seq_count = 0

        epoch_entries = list(train_fire_entries)
        random.shuffle(epoch_entries)

        # Accumulate mini-batches
        batch_X = []
        batch_y = []
        batch_m = []
        accum_count = 0

        for entry in epoch_entries:
            for X_seq, y_target, valid_mask in iter_fire_grid_sequences(
                entry, REPO_ROOT, channel_means, channel_stds, SEQ_LEN,
            ):
                seq_count += 1
                batch_X.append(X_seq)
                batch_y.append(y_target)
                batch_m.append(valid_mask)

                if len(batch_X) >= BATCH_SIZE:
                    xb = torch.from_numpy(np.stack(batch_X)).to(DEFAULT_DEVICE)
                    yb = torch.from_numpy(np.stack(batch_y)).unsqueeze(1).to(DEFAULT_DEVICE)
                    mb = torch.from_numpy(np.stack(batch_m)).unsqueeze(1).to(DEFAULT_DEVICE)
                    batch_X.clear()
                    batch_y.clear()
                    batch_m.clear()

                    logits = model(xb)
                    loss = criterion(logits, yb, mb) / GRAD_ACCUM_STEPS
                    loss.backward()
                    accum_count += 1

                    if accum_count >= GRAD_ACCUM_STEPS:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                        optimizer.step()
                        optimizer.zero_grad(set_to_none=True)
                        scheduler.step()
                        accum_count = 0

                    epoch_loss += loss.item() * GRAD_ACCUM_STEPS
                    epoch_steps += 1

                    with torch.no_grad():
                        pred = ((torch.sigmoid(logits) >= CLASSIFICATION_PROB_THRESHOLD) & (mb > 0.5)).int()
                        truth = (yb * mb).int()
                        mask_bool = mb > 0.5
                        tp += int(((pred == 1) & (truth == 1) & mask_bool).sum().cpu())
                        fp += int(((pred == 1) & (truth == 0) & mask_bool).sum().cpu())
                        fn += int(((pred == 0) & (truth == 1) & mask_bool).sum().cpu())
                        tn += int(((pred == 0) & (truth == 0) & mask_bool).sum().cpu())

        # Flush remaining accumulated gradients
        if accum_count > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        # Process leftover partial batch
        if batch_X:
            xb = torch.from_numpy(np.stack(batch_X)).to(DEFAULT_DEVICE)
            yb = torch.from_numpy(np.stack(batch_y)).unsqueeze(1).to(DEFAULT_DEVICE)
            mb = torch.from_numpy(np.stack(batch_m)).unsqueeze(1).to(DEFAULT_DEVICE)
            batch_X.clear()
            batch_y.clear()
            batch_m.clear()

            with torch.no_grad():
                logits = model(xb)
                loss_val = criterion(logits, yb, mb)
                epoch_loss += loss_val.item()
                epoch_steps += 1

                pred = ((torch.sigmoid(logits) >= CLASSIFICATION_PROB_THRESHOLD) & (mb > 0.5)).int()
                truth = (yb * mb).int()
                mask_bool = mb > 0.5
                tp += int(((pred == 1) & (truth == 1) & mask_bool).sum().cpu())
                fp += int(((pred == 1) & (truth == 0) & mask_bool).sum().cpu())
                fn += int(((pred == 0) & (truth == 1) & mask_bool).sum().cpu())
                tn += int(((pred == 0) & (truth == 0) & mask_bool).sum().cpu())

        avg_loss = epoch_loss / max(epoch_steps, 1)
        total = tp + fp + fn + tn
        acc = (tp + tn) / max(total, 1)
        prec = tp / max(tp + fp, 1)
        rec = tp / max(tp + fn, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-8)
        training_history.append({
            "epoch": epoch, "avg_loss": avg_loss, "accuracy": acc,
            "precision": prec, "recall": rec, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "final_lr": scheduler.get_lr(), "sequences": seq_count,
        })
        print(f"Epoch {epoch}/{EPOCHS}: loss={avg_loss:.4f} acc={acc:.4f} "
              f"prec={prec:.4f} rec={rec:.4f} f1={f1:.4f} lr={scheduler.get_lr():.2e} "
              f"seqs={seq_count:,}")

        # Save checkpoint after each epoch
        checkpoint = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "channel_means": channel_means.tolist(),
            "channel_stds": channel_stds.tolist(),
            "config": {
                "in_channels": IN_CHANNELS,
                "encoder_channels": ENCODER_CHANNELS,
                "bottleneck_channels": BOTTLENECK_CHANNELS,
                "seq_len": SEQ_LEN,
                "pad_h": PAD_H,
                "pad_w": PAD_W,
            },
            "training_history": training_history,
            "train_fires": [e["fire_name"] for e in train_fire_entries],
            "test_fires": [e["fire_name"] for e in test_fire_entries],
        }
        torch.save(checkpoint, ckpt_dir / "convgru_unet.pt")
        print(f"  -> checkpoint saved (epoch {epoch})")

    pd.DataFrame(training_history)

## 9) Evaluation at fixed threshold + per-fire breakdown

In [ ]:
def evaluate_convgru(model, entries, repo_root, channel_means, channel_stds,
                     seq_len, prob_threshold, batch_size, device):
    model.eval()
    tp = fp = fn = tn = 0
    n_eval = 0

    with torch.no_grad():
        for entry in entries:
            batch_X = []
            batch_y = []
            batch_m = []

            for X_seq, y_target, valid_mask in iter_fire_grid_sequences(
                entry, repo_root, channel_means, channel_stds, seq_len,
            ):
                batch_X.append(X_seq)
                batch_y.append(y_target)
                batch_m.append(valid_mask)

                if len(batch_X) >= batch_size:
                    xb = torch.from_numpy(np.stack(batch_X)).to(device)
                    yb = torch.from_numpy(np.stack(batch_y)).unsqueeze(1).to(device)
                    mb = torch.from_numpy(np.stack(batch_m)).unsqueeze(1).to(device)
                    batch_X.clear()
                    batch_y.clear()
                    batch_m.clear()

                    logits = model(xb)
                    probs = torch.sigmoid(logits)
                    pred = ((probs >= prob_threshold) & (mb > 0.5)).int()
                    truth = (yb * mb).int()
                    mask_bool = mb > 0.5

                    tp += int(((pred == 1) & (truth == 1) & mask_bool).sum().cpu())
                    fp += int(((pred == 1) & (truth == 0) & mask_bool).sum().cpu())
                    fn += int(((pred == 0) & (truth == 1) & mask_bool).sum().cpu())
                    tn += int(((pred == 0) & (truth == 0) & mask_bool).sum().cpu())
                    n_eval += int(mask_bool.sum().cpu())

            # Flush remaining
            if batch_X:
                xb = torch.from_numpy(np.stack(batch_X)).to(device)
                yb = torch.from_numpy(np.stack(batch_y)).unsqueeze(1).to(device)
                mb = torch.from_numpy(np.stack(batch_m)).unsqueeze(1).to(device)

                logits = model(xb)
                probs = torch.sigmoid(logits)
                pred = ((probs >= prob_threshold) & (mb > 0.5)).int()
                truth = (yb * mb).int()
                mask_bool = mb > 0.5

                tp += int(((pred == 1) & (truth == 1) & mask_bool).sum().cpu())
                fp += int(((pred == 1) & (truth == 0) & mask_bool).sum().cpu())
                fn += int(((pred == 0) & (truth == 1) & mask_bool).sum().cpu())
                tn += int(((pred == 0) & (truth == 0) & mask_bool).sum().cpu())
                n_eval += int(mask_bool.sum().cpu())

    return {
        "count": n_eval,
        "accuracy_overall": (tp + tn) / max(n_eval, 1),
        "positive_accuracy": tp / max(tp + fn, 1),
        "negative_accuracy": tn / max(tn + fp, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }


metrics_test = None

if RUN_EVALUATION_SECTION:
    metrics_test = evaluate_convgru(
        model, test_fire_entries, REPO_ROOT, channel_means, channel_stds,
        SEQ_LEN, CLASSIFICATION_PROB_THRESHOLD, BATCH_SIZE, DEFAULT_DEVICE,
    )
    prec = metrics_test["tp"] / max(metrics_test["tp"] + metrics_test["fp"], 1)
    rec = metrics_test["tp"] / max(metrics_test["tp"] + metrics_test["fn"], 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-8)
    print(f"Test pixels: {metrics_test['count']:,}")
    print(f"Overall accuracy: {metrics_test['accuracy_overall']:.4f}")
    print(f"Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
    print(f"TP={metrics_test['tp']:,}  FP={metrics_test['fp']:,}  "
          f"FN={metrics_test['fn']:,}  TN={metrics_test['tn']:,}")

In [ ]:
per_fire_results = []

if RUN_EVALUATION_SECTION and metrics_test is not None:
    print("Per-fire test metrics:")
    print(f"{'fire':>30s}  {'pixels':>12s}  {'prec':>6s}  {'rec':>6s}  {'f1':>6s}  {'TP':>8s}  {'FP':>8s}")
    print("-" * 90)

    for entry in test_fire_entries:
        fm = evaluate_convgru(
            model, [entry], REPO_ROOT, channel_means, channel_stds,
            SEQ_LEN, CLASSIFICATION_PROB_THRESHOLD, BATCH_SIZE, DEFAULT_DEVICE,
        )
        fm["fire_name"] = entry["fire_name"]
        prec = fm["tp"] / max(fm["tp"] + fm["fp"], 1)
        rec = fm["tp"] / max(fm["tp"] + fm["fn"], 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-8)
        fm["precision"] = prec
        fm["recall"] = rec
        fm["f1"] = f1
        per_fire_results.append(fm)
        print(f"{entry['fire_name']:>30s}  {fm['count']:>12,}  {prec:>6.4f}  {rec:>6.4f}  {f1:>6.4f}  {fm['tp']:>8,}  {fm['fp']:>8,}")

    per_fire_df = pd.DataFrame(per_fire_results)
    per_fire_df

## 10) Spatial prediction visualization

Compare predicted probability maps vs ground truth for sample hours from test fires.

In [ ]:
if RUN_VISUALIZATION_SECTION:
    model.eval()
    n_viz = 0
    max_viz = 6

    with torch.no_grad():
        for entry in test_fire_entries[:3]:
            ctx = load_fire_entry_context(entry, REPO_ROOT)
            H, W = ctx["goes_shape"]

            seq_count = 0
            for X_seq, y_target, valid_mask in iter_fire_grid_sequences(
                entry, REPO_ROOT, channel_means, channel_stds, SEQ_LEN,
            ):
                seq_count += 1
                # Show every 50th sequence to get variety
                if seq_count % 50 != 1:
                    continue

                xb = torch.from_numpy(X_seq[np.newaxis]).to(DEFAULT_DEVICE)
                logits = model(xb)
                prob_map = torch.sigmoid(logits).squeeze().cpu().numpy()

                # Crop to original grid size
                prob_crop = prob_map[:H, :W]
                truth_crop = y_target[:H, :W]
                mask_crop = valid_mask[:H, :W]

                fig, axes = plt.subplots(1, 3, figsize=(15, 4))

                im0 = axes[0].imshow(truth_crop * mask_crop, cmap="hot", vmin=0, vmax=1)
                axes[0].set_title(f"{entry['fire_name']} - Ground Truth")
                plt.colorbar(im0, ax=axes[0], fraction=0.046)

                im1 = axes[1].imshow(prob_crop * mask_crop, cmap="hot", vmin=0, vmax=1)
                axes[1].set_title("Predicted Probability")
                plt.colorbar(im1, ax=axes[1], fraction=0.046)

                pred_binary = ((prob_crop >= CLASSIFICATION_PROB_THRESHOLD) * mask_crop)
                im2 = axes[2].imshow(pred_binary, cmap="hot", vmin=0, vmax=1)
                axes[2].set_title(f"Predicted (thresh={CLASSIFICATION_PROB_THRESHOLD})")
                plt.colorbar(im2, ax=axes[2], fraction=0.046)

                fig.tight_layout()
                plt.show()

                n_viz += 1
                if n_viz >= max_viz:
                    break
            if n_viz >= max_viz:
                break

    print(f"Visualized {n_viz} spatial predictions.")

## 11) PR curve analysis

1. Train a validation model on inner-train fires, sweep thresholds on validation fires.
2. Transfer selected threshold to main model on test fires.
3. Sweep full test PR curve with main model.

In [ ]:
def compute_convgru_pr_curve(model, entries, repo_root, channel_means, channel_stds,
                             seq_len, thresholds, batch_size, device):
    model.eval()
    tp_arr = np.zeros(len(thresholds), dtype=np.int64)
    fp_arr = np.zeros(len(thresholds), dtype=np.int64)
    total_pos = 0
    total_neg = 0

    with torch.no_grad():
        for entry in entries:
            batch_X = []
            batch_y = []
            batch_m = []

            for X_seq, y_target, valid_mask in iter_fire_grid_sequences(
                entry, repo_root, channel_means, channel_stds, seq_len,
            ):
                batch_X.append(X_seq)
                batch_y.append(y_target)
                batch_m.append(valid_mask)

                if len(batch_X) >= batch_size:
                    xb = torch.from_numpy(np.stack(batch_X)).to(device)
                    yb_np = np.stack(batch_y)
                    mb_np = np.stack(batch_m)
                    batch_X.clear()
                    batch_y.clear()
                    batch_m.clear()

                    logits = model(xb)
                    probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()  # (B, H, W)

                    for b_idx in range(probs.shape[0]):
                        m = mb_np[b_idx] > 0.5
                        p_flat = probs[b_idx][m]
                        y_flat = yb_np[b_idx][m]
                        pos = y_flat > 0.5
                        total_pos += int(pos.sum())
                        total_neg += int((~pos).sum())
                        pred = p_flat[:, None] >= thresholds[None, :]
                        tp_arr += (pred & pos[:, None]).sum(axis=0).astype(np.int64)
                        fp_arr += (pred & (~pos)[:, None]).sum(axis=0).astype(np.int64)

            # Flush
            if batch_X:
                xb = torch.from_numpy(np.stack(batch_X)).to(device)
                yb_np = np.stack(batch_y)
                mb_np = np.stack(batch_m)

                logits = model(xb)
                probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()

                for b_idx in range(probs.shape[0]):
                    m = mb_np[b_idx] > 0.5
                    p_flat = probs[b_idx][m]
                    y_flat = yb_np[b_idx][m]
                    pos = y_flat > 0.5
                    total_pos += int(pos.sum())
                    total_neg += int((~pos).sum())
                    pred = p_flat[:, None] >= thresholds[None, :]
                    tp_arr += (pred & pos[:, None]).sum(axis=0).astype(np.int64)
                    fp_arr += (pred & (~pos)[:, None]).sum(axis=0).astype(np.int64)

    fn_arr = total_pos - tp_arr
    precision = safe_divide(tp_arr, tp_arr + fp_arr, default=1.0)
    recall = safe_divide(tp_arr, np.full_like(tp_arr, total_pos), default=0.0)
    f1 = safe_divide(2.0 * precision * recall, precision + recall, default=0.0)
    df = pd.DataFrame({
        "threshold": thresholds, "precision": precision, "recall": recall,
        "f1": f1, "tp": tp_arr, "fp": fp_arr, "fn": fn_arr,
        "tn": total_neg - fp_arr,
    })
    best = df.iloc[int(df["f1"].idxmax())]
    baseline = total_pos / max(total_pos + total_neg, 1)
    return {"df": df, "best": best, "baseline": baseline}


print("PR curve function defined.")

In [ ]:
VAL_PR_THRESHOLDS = np.linspace(0.0, 1.0, 201)
VAL_TUNED_THRESHOLD = None
val_pr_df = None
val_best = None
val_entries = []

if RUN_PR_SECTION:
    inner_train_entries, val_entries = split_validation_fire_entries(
        train_fire_entries, "auto", 0.30, 123,
    )
    print("inner-train fires:", [e["fire_name"] for e in inner_train_entries])
    print("validation fires:", [e["fire_name"] for e in val_entries])

    # Compute normalization for inner-train
    val_channel_means, val_channel_stds = compute_channel_stats(inner_train_entries)

    # Train validation model
    set_seed(SEED)
    val_model = ConvGRUUNet(
        in_channels=IN_CHANNELS,
        encoder_channels=ENCODER_CHANNELS,
        bottleneck_ch=BOTTLENECK_CHANNELS,
    ).to(DEFAULT_DEVICE)
    val_optimizer = torch.optim.AdamW(val_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    val_criterion = CombinedLoss(
        focal_weight=FOCAL_WEIGHT, dice_weight=DICE_WEIGHT,
        focal_alpha=FOCAL_ALPHA, focal_gamma=FOCAL_GAMMA,
    )

    val_est_steps = EPOCHS * est_steps_per_epoch
    val_scheduler = WarmupCosineScheduler(val_optimizer, int(LR_WARMUP_FRACTION * val_est_steps), val_est_steps)

    print("Training validation model...")
    for epoch in range(1, EPOCHS + 1):
        val_model.train()
        val_epoch_entries = list(inner_train_entries)
        random.shuffle(val_epoch_entries)
        batch_X = []
        batch_y = []
        batch_m = []

        for entry in val_epoch_entries:
            for X_seq, y_target, valid_mask in iter_fire_grid_sequences(
                entry, REPO_ROOT, val_channel_means, val_channel_stds, SEQ_LEN,
            ):
                batch_X.append(X_seq)
                batch_y.append(y_target)
                batch_m.append(valid_mask)

                if len(batch_X) >= BATCH_SIZE:
                    xb = torch.from_numpy(np.stack(batch_X)).to(DEFAULT_DEVICE)
                    yb = torch.from_numpy(np.stack(batch_y)).unsqueeze(1).to(DEFAULT_DEVICE)
                    mb = torch.from_numpy(np.stack(batch_m)).unsqueeze(1).to(DEFAULT_DEVICE)
                    batch_X.clear()
                    batch_y.clear()
                    batch_m.clear()

                    logits = val_model(xb)
                    loss = val_criterion(logits, yb, mb)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(val_model.parameters(), GRAD_CLIP_NORM)
                    val_optimizer.step()
                    val_optimizer.zero_grad(set_to_none=True)
                    val_scheduler.step()

        print(f"  val model epoch {epoch}/{EPOCHS} done")

    # Sweep validation PR curve
    val_pr_result = compute_convgru_pr_curve(
        val_model, val_entries, REPO_ROOT, val_channel_means, val_channel_stds,
        SEQ_LEN, VAL_PR_THRESHOLDS, BATCH_SIZE, DEFAULT_DEVICE,
    )
    val_pr_df = val_pr_result["df"]
    val_best = val_pr_result["best"].to_dict()
    VAL_TUNED_THRESHOLD = float(val_pr_result["best"]["threshold"])

    print(f"\nBest val threshold: {VAL_TUNED_THRESHOLD:.3f}")
    print(f"  precision={val_best['precision']:.4f} recall={val_best['recall']:.4f} f1={val_best['f1']:.4f}")

    fig, ax = plt.subplots(figsize=(7, 4))
    plot_df = val_pr_df.sort_values("recall")
    ax.plot(plot_df["recall"], plot_df["precision"], linewidth=2)
    ax.axhline(val_pr_result["baseline"], ls="--", color="gray",
               label=f"baseline ({val_pr_result['baseline']:.4f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Validation PR Curve (ConvGRU U-Net)")
    ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
train_threshold_test_metrics = None
test_pr_df = None
test_pr_best = None
test_pr_baseline = None

if RUN_PR_SECTION:
    if VAL_TUNED_THRESHOLD is not None:
        train_threshold_test_metrics = evaluate_convgru(
            model, test_fire_entries, REPO_ROOT, channel_means, channel_stds,
            SEQ_LEN, VAL_TUNED_THRESHOLD, BATCH_SIZE, DEFAULT_DEVICE,
        )
        m = train_threshold_test_metrics
        prec = m["tp"] / max(m["tp"] + m["fp"], 1)
        rec = m["tp"] / max(m["tp"] + m["fn"], 1)
        f1_val = 2 * prec * rec / max(prec + rec, 1e-8)
        print(f"Val threshold ({VAL_TUNED_THRESHOLD:.3f}) on test: prec={prec:.4f} rec={rec:.4f} f1={f1_val:.4f}")

    TEST_PR_THRESHOLDS = np.linspace(0.0, 1.0, 201)
    test_pr_result = compute_convgru_pr_curve(
        model, test_fire_entries, REPO_ROOT, channel_means, channel_stds,
        SEQ_LEN, TEST_PR_THRESHOLDS, BATCH_SIZE, DEFAULT_DEVICE,
    )
    test_pr_df = test_pr_result["df"]
    test_pr_best = test_pr_result["best"].to_dict()
    test_pr_baseline = float(test_pr_result["baseline"])

    print(f"\nBest test threshold: {test_pr_best['threshold']:.3f}")
    print(f"  precision={test_pr_best['precision']:.4f} recall={test_pr_best['recall']:.4f} f1={test_pr_best['f1']:.4f}")

    fig, ax = plt.subplots(figsize=(7, 4))
    plot_df = test_pr_df.sort_values("recall")
    ax.plot(plot_df["recall"], plot_df["precision"], linewidth=2)
    ax.axhline(test_pr_baseline, ls="--", color="gray",
               label=f"baseline ({test_pr_baseline:.4f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Test PR Curve (ConvGRU U-Net)")
    ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
if test_pr_df is not None:
    test_pr_df.sort_values("f1", ascending=False).head(20)

## 12) Report JSON

In [ ]:
def to_json_safe(obj):
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Series):
        return to_json_safe(obj.to_dict())
    return obj


if RUN_REPORT_SECTION:
    report = {
        "model": "convgru_unet",
        "target": "pixel_confidence_t_plus_1_binary",
        "prediction_type": "spatial_grid",
        "fires_used": [e["fire_name"] for e in fire_entries],
        "train_fires": [e["fire_name"] for e in train_fire_entries],
        "test_fires": [e["fire_name"] for e in test_fire_entries],
        "validation_fires": [e["fire_name"] for e in val_entries],
        "thresholds": {
            "positive_confidence": POSITIVE_THRESHOLD,
            "classification_probability": CLASSIFICATION_PROB_THRESHOLD,
            "validation_selected_probability": VAL_TUNED_THRESHOLD,
        },
        "split": {
            "method": "fire_holdout",
            "train_fire_count": len(train_fire_entries),
            "test_fire_count": len(test_fire_entries),
            "train_fire_fraction_target": FIRE_TRAIN_FRACTION,
            "split_seed": FIRE_SPLIT_SEED,
        },
        "architecture": {
            "type": "ConvGRU_UNet",
            "seq_len": SEQ_LEN,
            "in_channels": IN_CHANNELS,
            "encoder_channels": ENCODER_CHANNELS,
            "bottleneck_channels": BOTTLENECK_CHANNELS,
            "pad_h": PAD_H,
            "pad_w": PAD_W,
            "parameter_count": n_params,
        },
        "loss": {
            "type": "focal+dice",
            "focal_weight": FOCAL_WEIGHT,
            "dice_weight": DICE_WEIGHT,
            "focal_alpha": FOCAL_ALPHA,
            "focal_gamma": FOCAL_GAMMA,
        },
        "training": {
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "grad_clip_norm": GRAD_CLIP_NORM,
            "lr_warmup_fraction": LR_WARMUP_FRACTION,
            "seed": SEED,
            "device": str(DEFAULT_DEVICE),
            "history": training_history,
        },
        "channel_normalization": {
            "method": "per_channel_zscore_from_train_grids",
            "channel_names": CHANNEL_NAMES,
            "means": channel_means.tolist(),
            "stds": channel_stds.tolist(),
        },
        "metrics_test_fixed_threshold": {
            "threshold": CLASSIFICATION_PROB_THRESHOLD,
            **(metrics_test if metrics_test else {}),
        },
        "per_fire_test_metrics": per_fire_results if per_fire_results else [],
        "validation_threshold_selection": {
            "best_threshold": VAL_TUNED_THRESHOLD,
            "best_metrics": val_best,
        },
        "train_threshold_transfer": train_threshold_test_metrics,
        "test_pr_curve": {
            "baseline": test_pr_baseline,
            "best": test_pr_best,
            "top_by_f1": (
                test_pr_df.sort_values("f1", ascending=False)
                .head(12).to_dict(orient="records")
                if test_pr_df is not None else []
            ),
        },
        "grid_stats": {
            "train": train_grid_stats,
            "test": test_grid_stats,
        },
    }

    report = to_json_safe(report)
    out_dir = REPO_ROOT / "data" / "analysis" / "convgru_fire_holdout"
    out_dir.mkdir(parents=True, exist_ok=True)
    report_path = out_dir / "report.json"
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    print("saved:", report_path)
    print("Report keys:", list(report.keys()))

## 13) Save checkpoint

In [ ]:
import torch

ckpt_dir = REPO_ROOT / "data" / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

if RUN_TRAINING_SECTION and model is not None:
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "channel_means": channel_means.tolist(),
        "channel_stds": channel_stds.tolist(),
        "config": {
            "in_channels": IN_CHANNELS,
            "encoder_channels": ENCODER_CHANNELS,
            "bottleneck_channels": BOTTLENECK_CHANNELS,
            "seq_len": SEQ_LEN,
            "pad_h": PAD_H,
            "pad_w": PAD_W,
        },
        "train_fires": [e["fire_name"] for e in train_fire_entries],
        "test_fires": [e["fire_name"] for e in test_fire_entries],
    }
    torch.save(checkpoint, ckpt_dir / "convgru_unet.pt")
    print(f"Saved checkpoint to {ckpt_dir / 'convgru_unet.pt'}")
else:
    print("Skipped checkpoint save (training section did not run).")

## 13) Notes

- **Full spatial grid prediction**: Unlike previous models that predict per-pixel using flattened neighborhood features, this model predicts entire fire probability maps. This allows learning spatial patterns like wind-driven fire fronts and fire shapes.
- **ConvGRU at bottleneck**: Processing temporal dynamics at 14x24 resolution (1/8th of padded grid) keeps computation manageable while capturing fire spread patterns.
- **Skip connections from last timestep only**: The decoder uses encoder features from the most recent timestep, avoiding blurring across time while preserving fine spatial detail for the prediction.
- **Validity masking**: Fires have different grid sizes (32x59 to 105x187). Zero-padding to 112x192 with a validity mask channel ensures the model knows which pixels are real. Loss is computed only on valid pixels.
- **Focal + Dice loss**: Focal loss handles the extreme class imbalance (~0.3% positive rate) by down-weighting easy negatives. Dice loss provides a region-level F1-like signal that encourages spatial coherence in predictions.
- **No discounted rain**: The grid model uses 7 physical channels directly, without the discounted rain feature used in per-pixel models. Wind direction is encoded as sin/cos components.
- **Memory efficiency**: Sequences are streamed with a rolling buffer (same pattern as RNN notebook), not cached in memory. Each sample is ~4.8 MB (6 × 8 × 112 × 192 × 4 bytes).
- **Comparable evaluation**: Same fire-level holdout protocol (seed=42, 70/30 split) and evaluation metrics as LogReg (F1=0.716), MLP (F1=0.709), and GRU (F1=0.748) notebooks.